# Kronos-small exploratory smoke test

This notebook checks that the published checkpoint can forecast 20 daily bars for one AAPL date. It is a **technical smoke test**, not an estimate of prediction quality, a new blind test, or a Phase 8 score. It uses raw OHLCV, while the project's frozen target uses adjusted close. Run cells in order.


In [1]:
from pathlib import Path
import os, subprocess, sys, zipfile

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    root = Path('/content/HalalifyAI')
    if not root.exists():
        subprocess.run(['git', 'clone', 'https://github.com/frager098/HalalifyAI.git', str(root)], check=True)
    os.chdir(root)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch>=2.0', 'einops==0.8.1', 'huggingface_hub==0.33.1', 'safetensors==0.6.2', 'pandas', 'numpy'], check=True)
else:
    root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'AGENTS.md').is_file() and (p/'src').is_dir() and (p/'configs').is_dir()), None)
    if root is None:
        raise RuntimeError('Open this notebook from inside the HalalifyAI repository.')
    os.chdir(root)
print('Environment:', 'Colab' if IN_COLAB else 'local')
print('Repository root:', root)
print('Notebook kernel Python:', sys.executable)


Environment: local
Repository root: d:\Arham\FYP\HalalifyAI


In [2]:
import hashlib, json
contract = json.loads(Path('configs/phase8_evaluation_contract_v1.json').read_text(encoding='utf-8'))
source_info = contract['input_identities']['accepted_cleaned_csv']
prices_path = Path(source_info['path'])
print('Required input:', prices_path)
if IN_COLAB and not prices_path.is_file():
    from google.colab import files
    print('Upload kronos_small_inputs.zip containing the required file at the printed relative path.')
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith('.zip'):
        raise ValueError('Upload exactly one phase-specific ZIP file.')
    archive_path = Path(next(iter(uploaded)))
    with zipfile.ZipFile(archive_path) as bundle:
        members = bundle.infolist()
        expected = prices_path.as_posix()
        if [m.filename for m in members if not m.is_dir()] != [expected]:
            raise ValueError(f'ZIP must contain only {expected} at that exact path.')
        for member in members:
            target = (root / member.filename).resolve()
            if not target.is_relative_to(root.resolve()):
                raise ValueError(f'Unsafe ZIP path: {member.filename}')
            if target.exists() and not member.is_dir():
                raise FileExistsError(f'Will not overwrite {target}')
        bundle.extractall(root)
if not prices_path.is_file():
    raise FileNotFoundError(f'Missing {prices_path}. Place the accepted cleaned file at this exact repository-relative path.')
actual_hash = hashlib.sha256(prices_path.read_bytes()).hexdigest()
if actual_hash != source_info['sha256']:
    raise ValueError(f'Input hash differs from frozen contract: {actual_hash}. Use the accepted Phase 8 cleaned snapshot.')
print('Input SHA-256 matches frozen contract.')
import importlib.util
required_packages = ['numpy', 'pandas', 'torch', 'einops', 'huggingface_hub', 'safetensors']
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing_packages:
    raise RuntimeError(f'Missing packages in notebook kernel {sys.executable}: {missing_packages}. Select the project .venv kernel, or install into this exact Python with: {sys.executable} -m pip install torch einops==0.8.1 huggingface_hub==0.33.1 safetensors==0.6.2 pandas numpy')
print('Notebook kernel dependencies are available.')


Required input: data\processed\phase8_market_cleaned\phase8_alpaca_sip_20261124T005018Z_cleaned_v1\prices_clean.csv
Input SHA-256 matches frozen contract.


## Load fixed published code and checkpoint

The code comes from a fixed GitHub commit. The weights and tokenizer come from fixed Hugging Face revisions. The Git clone is kept in ignored interim data. Local users must first install the packages listed in `notebooks/README.md`.


In [3]:
UPSTREAM_COMMIT = '67b630e67f6a18c9e9be918d9b4337c960db1e9a'
MODEL_REVISION = '901c26c1332695a2a8f243eb2f37243a37bea320'
TOKENIZER_REVISION = '0e0117387f39004a9016484a186a908917e22426'
upstream = Path('data/interim/kronos_upstream').resolve()
if not upstream.exists():
    subprocess.run(['git', 'clone', 'https://github.com/shiyu-coder/Kronos.git', str(upstream)], check=True)
    subprocess.run(['git', '-C', str(upstream), 'checkout', '--detach', UPSTREAM_COMMIT], check=True)
commit = subprocess.check_output(['git', '-C', str(upstream), 'rev-parse', 'HEAD'], text=True).strip()
if commit != UPSTREAM_COMMIT:
    raise ValueError(f'Existing Kronos code is at {commit}, expected {UPSTREAM_COMMIT}. Use a fresh interim clone.')
sys.path.insert(0, str(upstream))
import random
import numpy as np
import torch
from model import Kronos, KronosTokenizer, KronosPredictor
SEED = 20261127
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('Upstream commit:', commit)
print('PyTorch:', torch.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = KronosTokenizer.from_pretrained('NeoQuasar/Kronos-Tokenizer-base', revision=TOKENIZER_REVISION)
model = Kronos.from_pretrained('NeoQuasar/Kronos-small', revision=MODEL_REVISION)
predictor = KronosPredictor(model, tokenizer, max_context=512)


ModuleNotFoundError: No module named 'numpy'

In [ ]:
import numpy as np
import pandas as pd
ticker, decision_date, lookback, horizon = 'AAPL', pd.Timestamp('2026-08-03'), 90, 20
cols = ['ticker', 'session_date', 'open', 'high', 'low', 'close', 'volume']
frame = pd.read_csv(prices_path, usecols=cols, parse_dates=['session_date'])
stock = frame.loc[frame.ticker.eq(ticker)].sort_values('session_date').reset_index(drop=True)
if stock.session_date.duplicated().any():
    raise ValueError('Duplicate AAPL session dates.')
matches = stock.index[stock.session_date.eq(decision_date)]
if len(matches) != 1:
    raise ValueError(f'{ticker} decision date {decision_date.date()} is unavailable.')
i = int(matches[0])
if i + 1 < lookback or len(stock) - i - 1 < horizon:
    raise ValueError('Insufficient historical context or later sessions for this smoke test.')
history = stock.iloc[i-lookback+1:i+1].copy()
future = stock.iloc[i+1:i+1+horizon].copy()
if not np.isfinite(history[['open','high','low','close','volume']].to_numpy(dtype=float)).all():
    raise ValueError('History contains missing or nonfinite OHLCV values.')
print('Historical input:', history.session_date.min().date(), 'through', history.session_date.max().date())
print('Forecast calendar:', future.session_date.min().date(), 'through', future.session_date.max().date())
print('The future table supplies dates only; no future prices enter the predictor.')


In [ ]:
if 'predictor' not in globals():
    raise RuntimeError('Kronos predictor is unavailable. Run the model-loading cell above and confirm that it finishes without an error before predicting.')
x = history[['open','high','low','close','volume']].reset_index(drop=True)
x_dates = history.session_date.reset_index(drop=True)
y_dates = future.session_date.reset_index(drop=True)
forecast = predictor.predict(df=x, x_timestamp=x_dates, y_timestamp=y_dates, pred_len=horizon, T=1.0, top_p=0.9, sample_count=1)
if len(forecast) != horizon or not np.isfinite(forecast['close'].to_numpy(dtype=float)).all():
    raise ValueError('Kronos returned an incomplete or nonfinite close forecast.')
predicted_return = float(forecast['close'].iloc[-1] / history['close'].iloc[-1] - 1)
actual_return = float(future['close'].iloc[-1] / history['close'].iloc[-1] - 1)
print('Predicted raw close 20-session return:', round(predicted_return, 5))
print('Observed raw close 20-session return:', round(actual_return, 5))
print('This one-stock check is diagnostic only; it is not the project return ranking or risk evaluation.')
display(forecast[['close']].head())


In [ ]:
import importlib.metadata as metadata
out_dir = Path('reports/model_evaluation/kronos_small_smoke_v1')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'{ticker}_{decision_date.date()}.json'
if out_path.exists():
    raise FileExistsError(f'Existing result preserved: {out_path}')
result = {'status':'exploratory_smoke_test', 'ticker':ticker, 'decision_date':str(decision_date.date()), 'horizon_sessions':horizon, 'lookback_sessions':lookback, 'input_path':str(prices_path), 'input_sha256':actual_hash, 'upstream_commit':commit, 'model_revision':MODEL_REVISION, 'tokenizer_revision':TOKENIZER_REVISION, 'seed':SEED, 'checkpoint':'NeoQuasar/Kronos-small', 'tokenizer':'NeoQuasar/Kronos-Tokenizer-base', 'torch_version':torch.__version__, 'huggingface_hub_version':metadata.version('huggingface_hub'), 'sampling':{'T':1.0,'top_p':0.9,'sample_count':1}, 'predicted_raw_close_return_20d':predicted_return, 'observed_raw_close_return_20d':actual_return, 'note':'One AAPL date using raw OHLCV; technical smoke test only. Not comparable to adjusted-close Phase 8 metrics or an untouched test.'}
out_path.write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')
print('Created:', out_path)
